In [64]:
#importing packages and env-var

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f

import os
from dotenv import load_dotenv

env = load_dotenv(dotenv_path='/mnt/d/linux/project/vars/.env')

#env vars for postgresql connection
mssqlsource = os.getenv('MSSQLSOURCE')
mssqluser = os.getenv('MSSQL_SPARK_USER')
mssqlpwd = os.getenv('MSSQL_SPARK_PWD')
mssqldriver = os.getenv('MSSQL_DRIVER')
sqlserverjar = os.getenv('SQLSERVER_JAR')
mssql_dwdest = os.getenv('MSSQL_CONN')

In [65]:
#load env-vars
pgconn = os.getenv('POSTGRE_CONN')
pgdriver = os.getenv('PG_DRIVER')
pgjar = os.getenv('POSTGRE_JAR')
pguser = os.getenv('POSTGRE_USER')
pgpwd = os.getenv('POSTGRE_PWD')

In [66]:
#create session 
spark = (( 
          SparkSession
          .builder
          .appName('invoices')
          .config('spark.jars', f'{sqlserverjar}')
          .config('spark.executor.extraClassPath', f'{sqlserverjar}')
          .config('spark.driver.extraClassPath', f'{sqlserverjar}')
          .getOrCreate()
        ))

In [67]:
#import invoice data from sales.invoice table

tmp_invoiceline_frame = ((spark
                            .read
                            .format('jdbc')
                            .option('url', mssqlsource)
                            .option('driver', mssqldriver)
                            .option('password', mssqlpwd)
                            .option('user', mssqluser)
                            .option('query', 
                                        
                                        """ 
                                            select
                                            
                                                invoiceid,
                                                invoicelineid,
                                                stockitemid,
                                                packagetypeid,
                                                quantity,
                                                cast(unitprice as decimal(12,2)) as unitprice,
                                                taxamount,
                                                extendedprice,
                                                lineprofit,
                                                cast(taxrate as decimal(12,2)) as taxrate
                                                
                                            from sales.invoicelines

                                        """
                                    )
                        )).load()

In [68]:
tmp_invoiceorder_frame = ((spark
                           .read
                           .format('jdbc')
                           .option('driver', mssqldriver)
                           .option('url', mssqlsource)
                           .option('user', mssqluser)
                           .option('password', mssqlpwd)
                           .option('query', 
                                    """ 
                                    
                                        select
                                        
                                            orderid,
                                            invoiceid,
                                            customerid,
                                            salespersonpersonid,
                                            accountspersonid,
                                            customerpurchaseordernumber,
                                            deliverymethodid,
                                            billtocustomerid,
                                            contactpersonid,
                                            packedbypersonid,
                                            invoicedate,
                                            cast(confirmeddeliverytime as date) as deliverydate,
                                            totaldryitems,
                                            totalchilleritems
                                            
                                        from sales.invoices
                                    
                                    """
                                   )
                        )).load()

In [69]:
#joining dataframes 

join_invoiceframes = (( tmp_invoiceline_frame
                        .join(tmp_invoiceorder_frame, 'invoiceid', 'left')
                            
                        ))

In [70]:
invoice = (( join_invoiceframes
                .select('invoiceid',
                        'invoicelineid',
                        'orderid',
                        'stockitemid',
                        'customerid',
                        'salespersonpersonid',
                        'accountspersonid',
                        'customerpurchaseordernumber',
                        'billtocustomerid',
                        'contactpersonid',
                        'packedbypersonid',
                        'packagetypeid',
                        'quantity',
                        'unitprice',
                        'taxamount',
                        'extendedprice',
                        'lineprofit',
                        'taxrate',
                        'totaldryitems',
                        'totalchilleritems'
                        )
                .withColumnRenamed('salespersonpersonid', 'salespersonid')
                .withColumnRenamed('customerpurchaseordernumber', 'purchaseordernumber')
        ))

In [71]:
((
    invoice
        .write
        .format('jdbc')
        .option('url', mssql_dwdest)
        .option('driver', mssqldriver)
        .option('user', mssqluser)
        .option('password', mssqlpwd)
        .option('dbtable', 'fact.invoices')
        .save(mode= 'overwrite')
))